# Offline training from Hub stores

Load the stores pushed by `01_collect_dataset.ipynb`, train a small DQN-style MOUSE model, and push the trained checkpoint to the Hugging Face Hub.

The model is built from three pieces: `StepEmbedder`, `Qwen3Backbone`, and a DQN head. `Qwen3Backbone` owns all pretrained config and weight loading, so the rest of the notebook reads `backbone.hidden_dim` from the constructed object.

In [1]:
import torch

from mouse_core.data import DataLoader, Datastore, load_stores_from_hub
from mouse_core.objectives import DqnObjectiveConfig, dqn_objective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead

# Hub dataset repo written by 01_collect_dataset.ipynb.
DATASET_ID = "mouse-example-dataset"

# Hub model repo where the trained checkpoint is uploaded.
MODEL_ID = "mouse-example-model"

# FrozenLake action count; used for the action embedding and DQN head width.
MAX_ACTIONS = 4

# FrozenLake 4x4 has 16 discrete observation states.
MAX_OBS_DISCRETE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load the collected stores

Load the named stores pushed by `01_collect_dataset.ipynb`, then combine them into one `Datastore` for sequential training batches.

In [2]:
stores = load_stores_from_hub(DATASET_ID, split="train")

store = Datastore()
store.append(stores)

for loaded_store in stores:
    print(loaded_store)
print(f"Combined training store: {store}")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.09k [00:00<?, ?B/s]

Datastore(name='frozenlake_deterministic', steps=200)
Datastore(name='frozenlake_slippery', steps=200)
Combined training store: Datastore(steps=400)


## Build model

This cell downloads the public `Qwen/Qwen3-0.6B` checkpoint unless it is already cached locally.

In [ ]:
backbone = Qwen3Backbone(
    pretrained="Qwen/Qwen3-0.6B",
    num_layers=2,
)
hidden_dim = backbone.hidden_dim

encoder = StepEmbedder(
    hidden_dim=hidden_dim,
    modalities=[
        {"name": "action", "embed": "discrete", "vocab_size": MAX_ACTIONS, "std": 0.02, "tokens": 1},
        {"name": "observation", "embed": "discrete", "vocab_size": MAX_OBS_DISCRETE, "std": 0.02, "tokens": 1},
        {"name": "reward", "embed": "rff", "std": 0.02, "in_min": 0.01, "in_max": 10.0, "tokens": 1},
        {"name": "done", "embed": "discrete", "vocab_size": 3, "std": 0.02, "tokens": 1},
        {"name": "scratch", "embed": "learnable", "tokens": 1, "required": False},
    ],
    concat_modalities=False,
)

heads = DiscreteActionValueHead(
    in_features=hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=hidden_dim,
    num_layers=1,
    scale=0.01,
)

# Compose the three independent pieces.
model = Model(
    encoder=encoder,
    backbone=backbone,
    heads=heads,
).train().to(device)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

## Train

In [4]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
dqn_cfg = DqnObjectiveConfig(weight=1.0, gamma=0.99, tau=0.005)

loader = DataLoader(
    store,
    sequence_length=8,
    batch_size=4,
    sampling="random",
    prefetch=2,
    num_workers=0,
)

for step in range(200):
    batch = loader.next_batch()
    out, step_stream, _ = model(batch)
    step_stream = step_stream.to(device)
    loss, metrics = dqn_objective(step_stream, out, dqn_cfg)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    model.polyak_update(action_value_tau=dqn_cfg.tau)
    if step % 10 == 0:
        print(f"step={step} loss={loss}")

loader.close()
print("Training finished.")

step=0 loss=4.42286764155142e-05
step=10 loss=0.0007249877089634538
step=20 loss=0.0001596109359525144
step=30 loss=0.00016423700435552746
step=40 loss=0.0003690158191602677
step=50 loss=0.0001392581034451723
step=60 loss=3.013961759279482e-05
step=70 loss=0.00027621653862297535
step=80 loss=4.994905975763686e-05
step=90 loss=0.0003890196676366031
step=100 loss=3.744801506400108e-05
step=110 loss=8.952054486144334e-05
step=120 loss=9.095937275560573e-05
step=130 loss=6.188663974171504e-05
step=140 loss=5.683307972503826e-05
step=150 loss=7.129206642275676e-05
step=160 loss=0.00024075384135358036
step=170 loss=8.922629604057875e-06
step=180 loss=1.3180526366340928e-05
step=190 loss=1.0923705303866882e-05
Training finished.


## Save to the Hugging Face Hub

Push the trained checkpoint under the hard-coded model name. The Hub client creates or updates the model repo under your authenticated Hugging Face account.

In [5]:
model.eval().to("cpu")
hub_url = push_model_to_hub(
    model,
    MODEL_ID,
    private=False,
    clear=True,
)
print(f"Uploaded model to {hub_url}")

Uploaded model to https://huggingface.co/micahr234/mouse-example-model
